# 1.Epoch,Batch and Step
- `Epoch`是`training_loop`的完成一轮`train`的单位，每一轮`Epoch`中样本顺序都会被随机打乱
- `Batch`是`train`中的单位，取决于`dataloader`对数据集的划分，每一轮`batch`输入一个(batch_size,seq_len,embedded_size)

$\text{每个Epoch包含的Batch数}=\text{len(dataloader)}$
- `Step`是模型参数更新的单位，取决于梯度累积`gradient_accumulation_batches`，`Optimizer`,`Scheduler`更关心`Step`

$$ n_{\text{step}} = \frac{n_{\text{batch}}}{\text{gradient\_accumulation\_steps}} $$

$$ \text{total\_step} = \frac{n_{\text{epoch}} \times \text{len}(\text{dataloader})}{\text{gradient\_accumulation\_steps}} $$

# 2.DeepSpeed
`DeepSpeed`有三个级别的优化阶段:
- `ZeRO-1`:优化器状态分片
- `ZeRO-2`:梯度状态分片
- `ZeRO-3`:模型参数分片

`ZeRO-2`只能用于训练，`ZeRO-3`可以用于推理

更多信息可以参考[blog](https://www.microsoft.com/en-us/research/blog/zero-deepspeed-new-system-optimizations-enable-training-models-with-over-100-billion-parameters/)

<div>  
  <img src="./assets/ZeRO.png" alt="Memory savings and communication volume for the three stages of ZeRO compared with standard data parallel baseline" style="width: 80%; height: auto;">  
</div>

# 2.1 Pre-Requests
install `DeepSpeed` version >=0.6.5

# 2.2 Accelerate DeepSpeed Plugin
1. use `accelerate config` to generate config file
2. use `accelerate launch my_script.py --args_to_my_script` to load config file for your script

# 2.3 Important code changes when using DeepSpeed Config File

# 3. DeepSpeed ZeRO原理

# 3.1 基本概念

# 3.1.1 通讯概念
1. `Reduce`
归约。常用的归约操作符有:求累加和SUM、求累乘积PROD、求最大值MAX等
<div>  
  <img src="./assets/reduce.png" alt="reduce" style="width: 50%; height: auto;">  
</div>

2. `Scatter`
散开。将数据切片再分发（broadcast）给所有节点
<div>  
  <img src="./assets/scatter.png" alt="scatter" style="width: 50%; height: auto;">  
</div>

1. `Reduce-Scatter`：小组分工。将数据先归约、再切片分发
<div>  
  <img src="./assets/reduce_scatter.png" alt="reduce_scatter" style="width: 50%; height: auto;">  
</div>

1. `All-Gather`：成果汇总。先把多个节点数据收集到一个主节点上，再把这个收集的数据分发到所有节点上
<div>  
  <img src="./assets/all_gather.png" alt="all_gather" style="width: 50%; height: auto;">  
</div>

# 3.1.2 Data Parallel
1. `Reduce-Scatter`计算每一个N等分后batch上的平均梯度
2. `All-Gather`汇总平均梯度

# 3.1.3 数据精度
- `FP16`(2Byte):sign(1)+exp(5)+frac(10)
- `BF16`(2Byte):sign(1)+exp(8)+frac(7)
- `FP32`(4Byte):sign(1)+exp(8)+frac(23)

`FP32`拥有最好的表示精度，但是占了4个字节，为了平衡表示能力与内存占用间的关系，`BP16`牺牲了精度，以获取与`FP32`相同的表示能力，因为动态范围大，所以相比`FP16`具有更好的数值稳定性

# 3.1.4 显存被什么占用了
<div>  
  <img src="./assets/memory.png" alt="memory" style="width: 20%; height: auto;">  
</div>

1. 神经网络参数：Parameters,gradients,activations（FP16）
2. Optimizer（Adam）:momentum,variance,Parameters（FP32）
3. 临时存储

\begin{aligned}
\text{Model States (bytes)} &= \overbrace{2 × \Psi}^{\text{Param}} + \overbrace{2 × \Psi}^{\text{Gradient}} + \overbrace{3 × 4 × \Psi}^{\text{Optimizer, 3 for Adam, 4 for FP32}} \\
&= 16 × Ψ
\end{aligned}

# 3.2 DeepSpeed ZeRO
# 3.2.1 ZeRO-1：优化器状态分区
将一个batch数据分为N份，将Optimizer State分为N份，每块GPU上存一份完整的模型参数

1. 梯度计算：每块 GPU 利用各自独一份的数据，做完一轮 forward 和 backward 后，各得到一份大小为 $\Psi$ 的梯度。
2. 梯度聚合：各自将大小为 $\Psi$ 的梯度平均分为 $N$ 份，经过一次 `Reduce-Scatter` 操作，使得不同显卡得到各自对应的一块聚合后的梯度。此过程产生单卡单向通信量 $\Psi$。
3. 参数更新：每张 GPU 利用各自手里的一小份优化器状态，更新对应的那一小份参数。
4. 参数同步：通过 `All-Gather` 操作从其他 GPU 中获取自身不具备的其他部分更新后的参数。此过程产生单卡单向通信量 $\Psi$。

显存占用量：$(2 + 2 + \frac{12}{N}) \Psi$ Bytes

单卡单向总通信量：$2\Psi$
# 3.2.2 ZeRO-2：梯度分区
反向传播之后，将梯度通过`Reduce-Scatter`操作，平均分为 $N$ 份存到各个节点上

与`ZeRO-1`区别在于，`ZeRO-2`在存储梯度时即通过`Reduce-Scatter`分片，而`ZeRO-1`在梯度聚合阶段才将存储的完整梯度数据分片

显存占用量：$(2 + \frac{2+12}{N}) \Psi$ Bytes

单卡单向总通信量：$2\Psi$
# 3.2.3 ZeRO-2：模型参数分区
将模型参数也分为N份

除了数据均分、优化器状态均分、梯度划分外，ZeRO-Stage 3 在 ZeRO-Stage 2 的基础上，将模型参数也进行了分块。每个GPU近维护各自对应的一部分。整个优化过程为：

1. 通过 `All-Gather` 操作从其他 GPU中获取自身不具备的模型参数，单卡单向通信量Ψ。
2. 利用各自独一份的数据，做完一轮foward，得到大小为Ψ Bytes 的 Activations，这时候不属于自己的其他参数块被释放，显存被 Activations 占用。
3. 通过 `All-Gather` 操作从其他 GPU中获取自身不具备的模型参数，单卡单向通信量Ψ。利用 Activations 与完整参数进行 backward。在这个过程中，各个GPU会得到不属于自己的梯度块，需要经过 `Reduce-Scatter`，动态地将其送给相应负责的GPU进行聚合（产生的单卡单向通信量为Ψ）；最终自己仅保留一份独属于自己的大小为 Ψ/N 的聚合后的梯度块。
4. 每块GPU利用自己的小块优化器与小块聚合梯度，更新自己的小块模型参数。

显存占用量是 (2+2+12)/N Ψ Bytes
单卡单向总通信量：$3\Psi$